# Assessment 1: Analysing historical data with system performance - Phase 2

**Student ID:** 35721588  
**Unit:** ITO5202  
**Teaching Period:** 5, 2026

**Dataset:** Brazilian E-Commerce Public Dataset by Olist  
**Source:** https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce

---

## Contents

**Part A: Analytical query design and implementation**
1. Business query design and justification
2. DataFrame API implementation
3. Spark SQL implementation
4. Result validation and API comparison

**Part B: System perspective and performance analysis**
1. Partitioning strategy analysis
2. Execution time benchmarking
3. Execution plan interpretation
4. DAG analysis via the Spark Web UI

---

## Environment and configuration

### Execution environment

For this project, we plan to run Spark in **local mode** on a single machine. In this setup, Spark does not create separate executor JVMs. Instead, the driver process handles the computation itself and uses the machine’s available logical CPU cores to run tasks in parallel. Because of this, the main memory setting that matters for our setup is spark.driver.memory, so we do not need to configure executor memory separately.

| Property | Value |
|---|---|
| Machine | MacBook Air (Retina, 13-inch, 2018) |
| Processor | 1.6 GHz dual-core Intel Core i5 |
| Logical cores | 4 |
| Physical memory | 8 GB |
| Operating system | macOS Sonoma 14.7.8 |
| Java | Eclipse Temurin JDK 17 (x64) |
| Python | 3.11.9 |
| PySpark | 3.5.1 |
| Spark master | `local[*]` |

The environment details are generated directly in the notebook rather than written in manually. This means the values referred to later in the Part B benchmarking discussion can be checked against the notebook output.

In [1]:
# Import packages
import os
import time
from statistics import median
import pandas as pd

from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark.sql.types import *

# Set up data directory folder path
DATA_DIR = "data"


---

## SparkSession configuration

For this project, there are three key Spark settings which we changed from their default values due to how they affect the behaviour we want to examine later in Part B.

**`spark.driver.memory = 3g`.** Our machine has 8 GB of RAM, which also needs to support the computer's other processes. Allocating too much memory to Spark could slow  down our execution significantly. This matters to us beyond a speed perspective, since Part B.2 compares execution times. More specifically, this would make our results less useful because they could reflect memory pressure rather than Spark's actual processing behaviour.

**`spark.sql.shuffle.partitions = 4`.** This setting determines how many partitions Spark creates after a shuffle. The default is 200, which makes more sense for a much larger cluster than for the local environment we are using here. With four available task slots and a dataset of around 113,000 rows at its largest, using 200 partitions would create many very small tasks and add unnecessary scheduling overhead.

**`spark.sql.adaptive.enabled = false`.** Spark's Adaptive Query Execution (AQE) can change the physical execution plan while a query is running. On one hand, this can improve performance, but on the other hand, it makes the execution harder to compare with the plan shown by `explain(extended=True)`. Because Parts B.3 and B.4 require us to examine the physical plan and its corresponding DAG, AQE is turned off so that the printed plan and the executed plan remain consistent. This also means that the partition counts discussed in Part B.1 reflect the values we set orselves rather than values Spark changes during execution.


In [2]:
# Spark session build with local mode, 4 task slots, AQE off (as explained above)
spark = SparkSession.builder \
    .appName("ITO5202-A1-Olist-Freight") \
    .master("local[*]") \
    .config("spark.driver.memory", "3g") \
    .config("spark.sql.shuffle.partitions", 4) \
    .config("spark.sql.adaptive.enabled", False) \
    .config("spark.sql.session.timeZone", "UTC") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
spark

26/09/20 15:47:49 WARN Utils: Your hostname, Mounishas-MacBook-Air.local resolves to a loopback address: 127.0.0.1; using 192.168.0.137 instead (on interface en0)
26/09/20 15:47:49 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/20 15:47:51 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
sc = spark.sparkContext

# Print the main Spark environment settings
print("Spark version:", spark.version)
print("Master:", sc.master)
print("Available task slots:", sc.defaultParallelism)
print("Driver memory:", spark.conf.get("spark.driver.memory"))
print("Shuffle partitions:", spark.conf.get("spark.sql.shuffle.partitions"))
print("AQE enabled:", spark.conf.get("spark.sql.adaptive.enabled"))
print("Broadcast join threshold (bytes):", spark.conf.get("spark.sql.autoBroadcastJoinThreshold"))

# Web UI link
print("Web UI:", sc.uiWebUrl)

Spark version: 3.5.1
Master: local[*]
Available task slots: 4
Driver memory: 3g
Shuffle partitions: 4
AQE enabled: false
Broadcast join threshold (bytes): 10485760b
Web UI: http://192.168.0.137:4040


In [4]:
# Confirms the session can run a job end to end
spark.range(10).count()

10


---

## Data Loading

### Defining schemas

Our nine source files are loaded in using manually defined schemas instead of `inferSchema=True`. This is because by us using schema inference, Spark has to inspect the data before it is able to load it, thus adding extra work (this is particularly important for the geolocation file, which contains over one million rows).

Furthermore, if we were to allow Spark to infer the schema automatically, the same column in different data files could be interpreted as being of different data types, which can have downstream impacts on analysis when we try to do joins. Instead, by defining the schema ourselves, we are able to ensure consistency in data types. 

We also note that as per our proposal, we are looking to only use seven of the nine available data files, as do not wish to include the payments and reviews dataset as they fall outside the scope of our analysis. As such, we do not read in these datasets such as to avoid unneccessary extra work.

#### _Note_

Given a heading error in Section 3 of our original proposal, we need to make a correction in our dataset list. The first table in Section 3 of our approved proposal is labelled `olist_geolocation_dataset.csv`, but the columns listed underneath it actually belong to `olist_order_items_dataset.csv`. The geolocation dataset then appears again later in the sme section of our proposal with the correct columns.

We note that the approved proposal has been kept unchanged in `proposal/proposal.md` for consistency. However, the schemas defined below use the actual, correct columns from each source file, which were checked against the downloaded dataset.

#### _Note 2_

The source Kaggle file uses the column names `product_name_lenght` and `product_description_lenght`. However, in our approved proposal, we listed these names with the correct spelling of "length". 

Now, for the purposes of our analysis, the schema needs to match the actual CSV headers exactly, thus creating a mismatch with our earlier proposal. To keep the rest of the notebook easier to read, these two columns are renamed immediately after loading.

In [5]:
# Define the schemas manually so Spark does not have to infer them

order_items_schema = StructType([
    StructField("order_id", StringType(), False),
    StructField("order_item_id", IntegerType(), False),
    StructField("product_id", StringType(), True),
    StructField("seller_id", StringType(), True),
    StructField("shipping_limit_date", TimestampType(), True),
    StructField("price", DoubleType(), True),
    StructField("freight_value", DoubleType(), True),
])

orders_schema = StructType([
    StructField("order_id", StringType(), False),
    StructField("customer_id", StringType(), False),
    StructField("order_status", StringType(), True),
    StructField("order_purchase_timestamp", TimestampType(), True),
    StructField("order_approved_at", TimestampType(), True),
    StructField("order_delivered_carrier_date", TimestampType(), True),
    StructField("order_delivered_customer_date", TimestampType(), True),
    StructField("order_estimated_delivery_date", TimestampType(), True),
])

customers_schema = StructType([
    StructField("customer_id", StringType(), False),
    StructField("customer_unique_id", StringType(), True),
    StructField("customer_zip_code_prefix", IntegerType(), True),
    StructField("customer_city", StringType(), True),
    StructField("customer_state", StringType(), True),
])

sellers_schema = StructType([
    StructField("seller_id", StringType(), False),
    StructField("seller_zip_code_prefix", IntegerType(), True),
    StructField("seller_city", StringType(), True),
    StructField("seller_state", StringType(), True),
])

geolocation_schema = StructType([
    StructField("geolocation_zip_code_prefix", IntegerType(), True),
    StructField("geolocation_lat", DoubleType(), True),
    StructField("geolocation_lng", DoubleType(), True),
    StructField("geolocation_city", StringType(), True),
    StructField("geolocation_state", StringType(), True),
])

products_schema = StructType([
    StructField("product_id", StringType(), False),
    StructField("product_category_name", StringType(), True),
    StructField("product_name_lenght", IntegerType(), True),
    StructField("product_description_lenght", IntegerType(), True),
    StructField("product_photos_qty", IntegerType(), True),
    StructField("product_weight_g", IntegerType(), True),
    StructField("product_length_cm", IntegerType(), True),
    StructField("product_height_cm", IntegerType(), True),
    StructField("product_width_cm", IntegerType(), True),
])

category_schema = StructType([
    StructField("product_category_name", StringType(), True),
    StructField("product_category_name_english", StringType(), True),
])

In [6]:
# Helper function to read a CSV
def load_csv(filename, schema):
    return (spark.read
            .option("header", True)
            .schema(schema)
            .csv(f"{DATA_DIR}/{filename}"))

order_items = load_csv("olist_order_items_dataset.csv", order_items_schema)
orders      = load_csv("olist_orders_dataset.csv", orders_schema)
customers   = load_csv("olist_customers_dataset.csv", customers_schema)
sellers     = load_csv("olist_sellers_dataset.csv", sellers_schema)
geolocation = load_csv("olist_geolocation_dataset.csv", geolocation_schema)
categories  = load_csv("product_category_name_translation.csv", category_schema)

# Correct the misspelled column names present in the source file headers
products = (load_csv("olist_products_dataset.csv", products_schema)
            .withColumnRenamed("product_name_lenght", "product_name_length")
            .withColumnRenamed("product_description_lenght", "product_description_length"))

In [7]:
# Check row counts against the expected values
order_items_count = order_items.count()
orders_count = orders.count()
customers_count = customers.count()
sellers_count = sellers.count()
geolocation_count = geolocation.count()
products_count = products.count()
categories_count = categories.count()

rows = [
    {
        "dataset": "order_items",
        "expected": 112650,
        "actual": order_items_count,
        "match": order_items_count == 112650,
        "columns": len(order_items.columns)
    },
    {
        "dataset": "orders",
        "expected": 99441,
        "actual": orders_count,
        "match": orders_count == 99441,
        "columns": len(orders.columns)
    },
    {
        "dataset": "customers",
        "expected": 99441,
        "actual": customers_count,
        "match": customers_count == 99441,
        "columns": len(customers.columns)
    },
    {
        "dataset": "sellers",
        "expected": 3095,
        "actual": sellers_count,
        "match": sellers_count == 3095,
        "columns": len(sellers.columns)
    },
    {
        "dataset": "geolocation",
        "expected": 1000163,
        "actual": geolocation_count,
        "match": geolocation_count == 1000163,
        "columns": len(geolocation.columns)
    },
    {
        "dataset": "products",
        "expected": 32951,
        "actual": products_count,
        "match": products_count == 32951,
        "columns": len(products.columns)
    },
    {
        "dataset": "categories",
        "expected": 71,
        "actual": categories_count,
        "match": categories_count == 71,
        "columns": len(categories.columns)
    }
]

pd.DataFrame(rows)

,dataset,expected,actual,match,columns
0,order_items,112650,112650,True,7
1,orders,99441,99441,True,8
2,customers,99441,99441,True,5
3,sellers,3095,3095,True,4
4,geolocation,1000163,1000163,True,5
5,products,32951,32951,True,9
6,categories,71,71,True,2



---

## Data quality assessment

Our approved proposal identified a few data quality issues that we will need to check before we can safely begin our analysis. 

As such, it is important for us to now look at these issues in the loaded data, measure how much of the data is affected, and consider the cleaning and aggregation choices we will need to make later when building our main analysis dataset.


In [8]:
# Check the different order statuses
orders.groupBy("order_status") \
      .count() \
      .orderBy(F.desc("count")) \
      .show()

+------------+-----+
|order_status|count|
+------------+-----+
|   delivered|96478|
|     shipped| 1107|
|    canceled|  625|
| unavailable|  609|
|    invoiced|  314|
|  processing|  301|
|     created|    5|
|    approved|    2|
+------------+-----+



In [9]:
# Check missing delivery-related timestamps
orders.select(
    F.sum(F.col("order_approved_at").isNull().cast("int")).alias("missing_approved"),
    F.sum(F.col("order_delivered_carrier_date").isNull().cast("int")).alias("missing_carrier"),
    F.sum(F.col("order_delivered_customer_date").isNull().cast("int")).alias("missing_delivered"),
    F.sum(F.col("order_estimated_delivery_date").isNull().cast("int")).alias("missing_estimated")
).show()

+----------------+---------------+-----------------+-----------------+
|missing_approved|missing_carrier|missing_delivered|missing_estimated|
+----------------+---------------+-----------------+-----------------+
|             160|           1783|             2965|                0|
+----------------+---------------+-----------------+-----------------+



In [10]:
# Check how many geolocation rows there are for each postcode prefix
geo_stats = geolocation.groupBy("geolocation_zip_code_prefix").count()

total_geo_rows = geolocation.count()
distinct_postcodes = geo_stats.count()

print("Total geolocation rows:", total_geo_rows)
print("Distinct postcode prefixes:", distinct_postcodes)

geo_stats.agg(
    F.avg("count").alias("average rows per postcode"),
    F.max("count").alias("maximum rows per postcode")
).show()

Total geolocation rows: 1000163
Distinct postcode prefixes: 19015


+-------------------------+-------------------------+
|average rows per postcode|maximum rows per postcode|
+-------------------------+-------------------------+
|       52.598632658427555|                     1146|
+-------------------------+-------------------------+



In [11]:
# Number of different customer postcode prefixes
customer_postcodes = customers.select("customer_zip_code_prefix").distinct().count()

print("Distinct customer postcode prefixes:", customer_postcodes)


# Check how customers are distributed across states
customers.groupBy("customer_state") \
         .count() \
         .orderBy(F.desc("count")) \
         .show(10)

Distinct customer postcode prefixes: 14994
+--------------+-----+
|customer_state|count|
+--------------+-----+
|            SP|41746|
|            RJ|12852|
|            MG|11635|
|            RS| 5466|
|            PR| 5045|
|            SC| 3637|
|            BA| 3380|
|            DF| 2140|
|            ES| 2033|
|            GO| 2020|
+--------------+-----+
only showing top 10 rows



In [12]:
# Check for missing product information used later in the analysis

products.select(
    F.sum(F.col("product_category_name").isNull().cast("int")).alias("missing_category"),
    F.sum(F.col("product_weight_g").isNull().cast("int")).alias("missing_weight"),
    F.sum(F.col("product_length_cm").isNull().cast("int")).alias("missing_length"),
    F.sum(F.col("product_height_cm").isNull().cast("int")).alias("missing_height"),
    F.sum(F.col("product_width_cm").isNull().cast("int")).alias("missing_width")
).show()

+----------------+--------------+--------------+--------------+-------------+
|missing_category|missing_weight|missing_length|missing_height|missing_width|
+----------------+--------------+--------------+--------------+-------------+
|             610|             2|             2|             2|            2|
+----------------+--------------+--------------+--------------+-------------+




---

## Data quality findings

**Order completeness:** There are 99,441 orders in total, with 96,478 marked as `delivered` (97.0%). However, 2,965 orders have missing `order_delivered_customer_date`, which exceeds the number of non-delivered orders by eight. This means that in our data, there are eight orders which were marked as delivered even though there was no recorded delivery timestamp. Furthermore, there is no indication from the data as to what could be causing this (e.g. delivery error, system issue, , etc.).
Since we do not have a clear cause, we should treat this as a data integrity issue, and thus want to filter our analysis for both `oder_status = 'delivered'`, as well as a non-null delivery timestamp. 

**Geolocation duplication:** The geolocation dataset has 1,000,163 rows, however it only contains 19,015 unique postcode prefixes. That means that on average, each unique postcode appears ~52.6 times in the dataset, with the most common postcode appearing 1,146 times. Given this, if we were to join this dataset directly, it would result in a large number of duplicate matches.
Thus, we want to reduce our geolocation data to one row per postcode prefix before conducting any joins. As an additional benefit, at this smaller size, it is also small enough to be used as a broadcast lookup rather than requiring both sides of the join to be shuffled.

**Cardinality of partitioning column:** `customer_zip_code_prefix` contains 14,994 unique values, which we can see is very close to the ~15,000 expected values we mentioned in the proposal. Another thing we briefly mentioned in the proposal was the uneven distribution of customers across states. 
From our above data exploration, we can see a much clearer picture of the customer distribution, with São Paulo containing 41,746 customers, which accounts for ~42% of the total, while SP, RJ and MG together account for aother ~66.6%. 
This uneven distribution is important for us later when we consider the hash and range partitioning comparison in Part B.1, because range partitioning may produce less balanced partitions when the values themselves are unevenly distributed.

**Product attribute completeness:** We can see that there are 610 products (1.9%) with no category name, while two products are missing physical dimension values. This means measures that depend on package volume or weight cannot be calculated for those rows. 
As such, instead of filling in estimated values, we want to exclude these rows from the relevant weight-based calculations, with the number of affected records being reported with the results.


---

## Part A.1: Business Query Design

### Analytical Question

When considering any analysis, it is important for us to first consider the business context of the data. 

We know that Olist operates as a marketplace intermediary; meaning that sellers are able to set their own prices vand Olist charges the customer a freight amount per item. However, what we aren't able to immediately ascertain is how to compare these values, any indication of freight profit, cost recovery or whether a shipment was priced correctly.That is to say, the same freight charge can represent something quite different for a light product and a heavy product, or for shipments travelling between different states.

Thus, for our analysis, freight is therefore compared relative to product weight and shipping lane.

From here, we seek to understand the answer to one key question through our query:

> **For each shipping lane (seller state to customer state) and each calendar quarter, what is the mean freight charged per kilogram, how does it compare to other lanes active in the same quarter, and how has it changed from the same lane's previous period?**

In order for us to answer this, our analysis first groups order items by shipping lane and quarter so that we have final output contains one row for each qualifying lane and quarter. We then want to calculate freight charged per kilogram, rank the qualifying lanes within each quarter, and compare each lane with its own result from the previous quarter.

#### **How we can interpret the results**

Our query can provide us with two different types of comparison:

- The **within-quarter** ranking, which compares shipping lanes with one another. We note that this needs to be interpreted carefully because freight charges are affected by distance and other shipment characteristics. For example, a shipment within one state is not directly comparable with one travelling across Brazil. The ranking is therefore used mainly to show the distribution of freight per kilogram and identify lanes at the higher or lower ends of that distribution.
- The **within-lane** comparison, which is useful for looking at change over time. Because the seller and customer states remain the same with lanes, geography is more consistent when a lane is compared with its own earlier result. A large change may therefore suggest a change in factors such as the products being shipped, package size, carrier arrangements or freight pricing.

Importantly, we need to keep in mind that a higher or lower value is not treated as automatically "good" or "bad". Rather, our main purpose should be the identification of patterns that may be worth investigating further.

#### **What the analysis can tell us**

The result of our analysis can be leverage as a potential screening tool which can highlight lanes that:

- Have relatively high or low freight per kilogram within a quarter; or
- Show a noticeable change compared with their previous observed quarter.

The main purpose of the analysis is therefore to reduce a large number of individual order items into a smaller set of lane-quarter results that can be investigated more closely.

#### **Operations used**

From the above, it becomes clear that the results we need cannot be produced using a single aggregation. This is due to the fact that the required information is spread across several datasets and the analysis includes both group-level and time-based comparisons.

| Operation | Use in the analysis |
|---|---|
| **Joins** | `order_items` is combined with `orders` for order dates and status, `customers` and `sellers` for the shipping lane, and `products` for product weight. |
| **Derived columns** | Shipping lane, weight in kilograms, freight per kilogram and purchase quarter are calculated from the source columns. |
| **Aggregation** | Order items are grouped by shipping lane and quarter to produce the final lane-quarter level measures. |
| **Post-aggregation filtering** | Lane-quarter groups with too few items are removed after aggregations due to the fact that the minimum count applies to the group rather than an individual row. |
| **Window functions** | `RANK()` compares lanes within the same quarter, while `LAG()` retrieves the previous available result for each lane. |
| **Time-based analysis** | Calendar quarters are derived from `order_purchase_timestamp` and used to compare results over time. | 

We can see from the above that our query plans to use six of the operation types listed in the assessment requirements, which is above the required minimum.

#### **Why Spark is appropriate for the analysis**

It is obvious that our chosen dataset is small enough to run on one machine, and thus distributed computing is not neccessarily a requirement purely driven by dataset size. 

In saying that, Spark is still useful because the query contains several operations that are important in distributed data processing. Our analysis combines datasets with different sizes, performs joins on high-cardinality keys, aggregates the resulting records, and then applies window functions that require the data to be organised in different ways. 

Some of these operations can require Spark to redistribute data between partitions. The value of Spark here is therefore not that the dataset could not be processed without it, but that the query demonstrates the types of operations and execution decisions that become important with larger distributed datasets.

#### **Filtering decisions**

- **Delivered orders only:** Our data quality checks in the previous section of tis notebook showed that our dataset contained eight orders which were marked as `delivered` despite having no recorded customer delivery timestamp. Thus, our analysis requires both `order_status = 'delivered'` and a non-null `order_delivered_customer_date`. 
- **Recorded product weight:** Freight per kilogram cannot be calculated when `product_weight_g` is missing. As such, records where `product_weight_g` is missing are excluded from calculations that require weight rather than simply estimating a value for them. 
- **Minimum lane-quarter volume:** Lane-quarter groups containing only a small number of items can produce unstable or extreme results. Thus, a minimum item count is therefore applied after aggregation so the ranking is based on groups with a more meaningful amount of data.


---

## Part A.2: DataFrame API implementation

### Building the analytical base view

The information we need for our analysis is spread across several source files, so the first step for us is to combine them into a single row-level DataFrame.

This base view brings together the order, customer, seller and product information needed for our query, and also applies the filtering decisions we identified earlier.

The remaining derived columns, such as shipping lane, product weight in kilograms and purchase quarter, are also created at this stage so that we can leverage them later in our analysis.

The completed base view is cached because it is used repeatedly in later sections, including the DataFrame and SQL versions of the query, the equivalence check, partitioning experiments and execution-plan analysis. This avoids rebuilding the same joins and derived columns each time.

#### **Join approach**

Since our source tables are not all the same size, the joins need to be handled differently depending on the data involved. 

The larger joins, particularly those involving `order_items`, `orders` and `customers`, use high-cardinality keys and may require Spark to redistribute records between partitions. This makes them more expensive than joins involving the smaller lookup tables. 

The smaller datasets on the other hand, such as sellers, products and the reduced geolocation lookup, are small enough to be candidates for broadcast joins, which avoids shuffling both sides.

In particular, we had earlier discussed the pre-processing work required for the geolocation dataset and outlined the potential for duplicate matching due to multiple rows containing the same postcode prefix. In the context of our join approach, we can see that this additional processing step has the added benefit of making the lookup much smaller, and thus making it suitable for a broadcast join as well.


In [13]:
# Reduce the geolocation data to one row for each postcode prefix
geo_lookup = geolocation.groupBy("geolocation_zip_code_prefix").agg(
    F.avg("geolocation_lat").alias("lat"),
    F.avg("geolocation_lng").alias("lng")
)

# Cache the geolocation data
geo_lookup.cache()

geo_count = geo_lookup.count()
print("Geolocation lookup rows:", geo_count)

Geolocation lookup rows: 19015


In [14]:
# Keep orders that were delivered and have a delivery date
orders_delivered = orders.filter(
    (F.col("order_status") == "delivered") &
    (F.col("order_delivered_customer_date").isNotNull())
)

delivered_count = orders_delivered.count()
print("Delivered orders retained:", delivered_count)

Delivered orders retained: 96470


In [15]:
# Only keep the columns needed from each table for the joins
orders_join = orders_delivered.select(
    "order_id",
    "customer_id",
    "order_purchase_timestamp",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
)

customers_join = customers.select(
    "customer_id",
    "customer_state",
    "customer_zip_code_prefix"
)

sellers_join = sellers.select(
    "seller_id",
    "seller_state",
    "seller_zip_code_prefix"
)

products_join = products.select(
    "product_id",
    "product_category_name",
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
)

In [16]:
# Build the main row-level dataset
base = order_items.join(
    orders_join,
    on="order_id",
    how="inner"
)

base = base.join(
    customers_join,
    on="customer_id",
    how="inner"
)

# These tables are small, so request broadcast joins
base = base.join(
    F.broadcast(sellers_join),
    on="seller_id",
    how="inner"
)

base = base.join(
    F.broadcast(products_join),
    on="product_id",
    how="left"
)

print("Base view rows after joins:", base.count())

Base view rows after joins: 110189


In [17]:
# Separate postcode lookups for customers and sellers
customer_geo = geo_lookup.select(
    F.col("geolocation_zip_code_prefix").alias("customer_zip_code_prefix"),
    F.col("lat").alias("cust_lat"),
    F.col("lng").alias("cust_lng")
)

seller_geo = geo_lookup.select(
    F.col("geolocation_zip_code_prefix").alias("seller_zip_code_prefix"),
    F.col("lat").alias("sell_lat"),
    F.col("lng").alias("sell_lng")
)

In [18]:
# Add approximate coordinates for both ends of the shipping lane
base = base.join(
    F.broadcast(customer_geo),
    on="customer_zip_code_prefix",
    how="left"
)

base = base.join(
    F.broadcast(seller_geo),
    on="seller_zip_code_prefix",
    how="left"
)

In [19]:
# Create the columns needed for analysis
base_view = base.withColumn(
    "lane",
    F.concat_ws(" -> ", F.col("seller_state"), F.col("customer_state"))
)

base_view = base_view.withColumn(
    "purchase_quarter",
    F.concat_ws(
        "-",
        F.year("order_purchase_timestamp"),
        F.concat(F.lit("Q"), F.quarter("order_purchase_timestamp"))
    )
)

base_view = base_view.withColumn(
    "weight_kg",
    F.col("product_weight_g") / 1000.0
)

base_view = base_view.withColumn(
    "freight_per_kg",
    F.when(
        F.col("weight_kg") > 0,
        F.col("freight_value") / F.col("weight_kg")
    )
)

base_view = base_view.withColumn(
    "freight_to_price_ratio",
    F.when(
        F.col("price") > 0,
        F.col("freight_value") / F.col("price")
    )
)

base_view = base_view.withColumn(
    "package_volume_cm3",
    F.col("product_length_cm") *
    F.col("product_height_cm") *
    F.col("product_width_cm")
)

In [20]:
# Approximate straight-line distance between the seller and customer
base_view = base_view.withColumn(
    "distance_km",
    F.lit(6371.0) * 2 * F.asin(
        F.sqrt(
            F.pow(
                F.sin(
                    F.radians(F.col("cust_lat") - F.col("sell_lat")) / 2
                ),
                2
            )
            +
            F.cos(F.radians(F.col("sell_lat"))) *
            F.cos(F.radians(F.col("cust_lat"))) *
            F.pow(
                F.sin(
                    F.radians(F.col("cust_lng") - F.col("sell_lng")) / 2
                ),
                2
            )
        )
    )
)

base_view = base_view.withColumn(
    "is_intrastate",
    F.col("seller_state") == F.col("customer_state")
)

In [21]:
# Keep the columns that we will later
base_view = base_view.select(
    "order_id",
    "order_item_id",
    "lane",
    "seller_state",
    "customer_state",
    "customer_zip_code_prefix",
    "seller_zip_code_prefix",
    "purchase_quarter",
    "order_purchase_timestamp",
    "price",
    "freight_value",
    "weight_kg",
    "freight_per_kg",
    "freight_to_price_ratio",
    "package_volume_cm3",
    "distance_km",
    "is_intrastate",
    "product_category_name"
)

In [22]:
# Confirm how many delivered orders lack a delivery timestamp
orders.filter(
    (F.col("order_status") == "delivered")
    & F.col("order_delivered_customer_date").isNull()
).count()

8